# **<p align="center"> Apo Structure SoakDB Metadata </p>**

## <ins> Description </ins>
### The goal of this notebook is to walk over the key soakdb.ipynb tags / metadata identified as eligible to identify apo structure pdbs.

<details open>
<summary> <font size = "6"> <ins> Rationale </ins> </font> </summary>

- ***Dimple VS refine.split.ground-state*** (dimple vs refmac refinement)

    - dimple.pdb 

- All Dimples VS PanDDA no ligand present

    - Descritption: This is the two extremes - barely no curation vs very curated.

    - All Dimple: 

        - Since binding is weak, dimple will not be able to identify the holo stucture, and so it is fine to use the dimple structure as an apo structure even if the ligand binds

    - PanDDA no ligand present

        - If the previous assumption is wrong, then the only way to ensure that the ligand is not bound, is if it is labelled by the "User" while using PanDDA

        - Details:

            - Note that for the same crystal, PanDDA can be run multiple times. Some times this will output a "clear" no ligand bound result, and sometimes a "ligand bound" result

</details>
<br>
<details open>
<summary> <font size="6"> <ins>How to Identify Apo Structures </ins> </font> </summary>

- Metadata:

    - They stopped being processed during the XChem Pipeline

        - At what stage did they stop?

            - At PanDDA -> You only move past PanDDA if the curator "says" that a ligand has been identified to bind.
- Data

    - `dimple.pdb` / `final.pdb` inside dimple file -> should be updated with the latest dimple version 

    - `{datasetID}-panda-input.pdb` or `dimple.pdb` in main directory -> 


</details>

</br>

<details >
<summary> <font size = "6"> <ins> Key Points 2 </ins> </font> </summary>

- `bulletpoint1`


- `bulletpoint2`

- [link]()

- image1: <img src="" width = 50%>



</details>

</br>


In [59]:
# Load Key Libraries
from pathlib import Path
rootdir = Path("../../../..").resolve()
import sys
sys.path.insert(0, str(rootdir) )
from typing import Callable, Generator
#### General Purpose Libraries
# import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shutil import copy2
#### Database Libraries
from sqlalchemy import create_engine, inspect
#### Molecular Libraries
import gemmi 
# from rdkit import Chem
# import parasail
##### Local Libraries
from xaidar.data.molecModels import loadPDB
from xaidar.data.molecModels import get_pdb_stats,  get_res_CoM , get_atom_coord
from xaidar.data.molecModels import flatten_pdb, createPDB, clear_empty
from xaidar.data.molecModels import sele_pdb, sele_Lig, sele_AA, sele_chain_name

In [2]:
# Test python objs
test_pdb_path = rootdir.joinpath("data/ev2a/fragalysis/02-chemofint/reference/A0926a.pdb")
test_pdb = loadPDB(test_pdb_path)
test_prot = sele_pdb( test_pdb, sele_AA)
test_prot_mon = sele_pdb( test_prot, sele_chain_name, chain_name = "A")
test_lig = sele_pdb( test_pdb, sele_Lig)
test_chain = flatten_pdb( test_prot_mon, "chain" )
test_res = test_chain[0][0] 
test_atom =  test_res[0]

---

## Bullet point 1

- ### Subpoint 1.1

In [3]:
# Load the .sqlite file
dls_path = Path("/dls/labxchem/data/")
ev2a_dls_path = dls_path.joinpath("lb32627/lb32627-66/")
sqliteFilePath = ev2a_dls_path.joinpath( "processing/database/soakDBDataFile.sqlite").resolve() # Path to your .sqlite file
print("Sqlite File Path: {}".format( sqliteFilePath ) )

Sqlite File Path: /dls/labxchem/data/lb32627/lb32627-66/processing/database/soakDBDataFile.sqlite


In [4]:
# load_sqlite_to_dict() -> Function to load specified tables from a SQLite database into a dictionary of pandas DataFrames
def load_sqlite_to_dict(sqlite_file_path: Path, tables: list[str] | None = None) -> dict[str, pd.DataFrame]:
    """
    Load specified tables from a SQLite database into a dictionary of pandas DataFrames.

    Parameters:
    - sqlite_file_path: Path to the SQLite database file.
    - tables: List of table names to load.

    Returns:
    - A dictionary where keys are table names and values are corresponding DataFrames.
    """
    # Step 1: Create an SQLAlchemy engine to connect to the SQLite database
    engine = create_engine(f"sqlite:///{sqliteFilePath}")

    # # Step 2: Inspect the database to see available tables
    # inspector = inspect(engine)
    # tables = inspector.get_table_names()
    # print("Tables in the database:")
    # print(tables)

    # Step 3: Load a specific table into a pandas DataFrame
    if tables is None:
        tables = ['depositTable', 'mainTable', 'panddaTable', 'soakDB']  # ['mainTable', 'panddaTable', 'collectionTable', 'depositTable','soakDB', 'zenodoTable','Pucks', ]
    loadtable = lambda table_name : pd.read_sql(f"SELECT * FROM {table_name}", con=engine)
    tabledf_dict = {table_name : loadtable(table_name) for table_name in tables[:4]  }

    # for df in tabledf_dict.values():
    #     filter_colms = [ colm for colm in df.columns if not all( df[colm].isna() ) ]    # Show columns with data
    #     display( filter_colms)
        # display( df] )
    # tabledf_lst = maindf, panddadf, collectdf, depoistdf 
    # print(type( tabledf_dict.values()))

    # display( list( tabledf_dict.values() )[0].iloc[:,:3])



    # Step 6: Close the connection (optional, as SQLAlchemy handles it automatically)
    engine.dispose()
    return tabledf_dict


In [7]:
# Load the tables into a dictionary of DataFrames
tabledf_dict = load_sqlite_to_dict(sqliteFilePath )

In [9]:
# Access individual DataFrames from the dictionary
main_df = tabledf_dict["mainTable"].copy()
pandda_df = tabledf_dict["panddaTable"].copy()
# collect_df = tabledf_dict["collectionTable"].copy()

## Main Table Tags

In [ ]:
# Unique Identifiers
cryst_name = main_df["CrystalName"] # Unique identifier for each crystal E.g. CHIKV_MacB-x0270


# Data Processing Stages
collectedMainDF = main_df[ 
                    # main_df[ "HarvestStatus"] == "done"  & 
                    # main_df[ "MountingResult"][:2] == "OK" & 
                    main_df[ "DataCollectionOutcome" ] == "success"] 

    # Dimple
dimpleMainDF = collectedMainDF[
                    collectedMainDF[ "DataProcessingDimpleSuccessful" ] == "TRUE" &
                     collectedMainDF[ "DimpleStatus" ] == "finished"
                                    ]

failedDimpleMainDF = collectedMainDF[
                    collectedMainDF[ "DataProcessingDimpleSuccessful" ] != "TRUE" 
]

    # PanDDA
panddaMainDF = dimpleMainDF[
                    dimpleMainDF[ "DimplePANDDAwasRun" ] == "TRUE" &
                    dimpleMainDF[ "DimplePANDDAreject" ] == "FALSE"
                            ]
failedPanddaMainDF = dimpleMainDF[
                    dimpleMainDF[ "DimplePANDDAwasRun" ] == "TRUE" &
                    dimpleMainDF[ "DimplePANDDAreject" ] != "FALSE"
                            ]

hitPanddaMainDF = panddaMainDF[ 
                        panddaMainDF[ "DimplePANDDAhit"] == "TRUE"
                        ]

noHitPanddaMainDF = panddaMainDF[ 
                        panddaMainDF[ "DimplePANDDAhit"] != "TRUE"
                        ]

filteredMainDF = dimpleMainDF

# Paths of Interest
dimplePathRefPDB = filteredMainDF[ "DimpleReferencePDB" ]
dimplePathMTZ = filteredMainDF[ "DimplePathToMTZ" ]
dimplePathPDB = filteredMainDF[ "DimplePathToPDB" ]

# Experimental Quality Metrics
    # Dimple
dimpleRcrys = filteredMainDF[ "DimpleRcryst" ]
dimpleRfree = filteredMainDF[ "DimpleRfree" ]



## All Dimples

## PanDDA No Ligand Bound

- PanDDA Table

In [22]:
no_lig_df =  pandda_df.groupby("CrystalName").filter( lambda x: (x["PANDDA_site_confidence"] == "0 - no ligand present" ).all() )  
no_lig_df.shape

(9, 38)

In [23]:
no_lig_df["CrystalName"]

49      A71EV2A-x0582
256     A71EV2A-x0131
1536    A71EV2A-x1932
1554    A71EV2A-x2441
1574    A71EV2A-x2454
1584    A71EV2A-x2378
1585    A71EV2A-x2374
1595    A71EV2A-x2769
1631    A71EV2A-x2959
Name: CrystalName, dtype: object

In [24]:
main_df[main_df["CrystalName"].isin(no_lig_df["CrystalName"])]


,ID,LabVisit,LibraryPlate,SourceWell,LibraryName,CompoundSMILES,CompoundCode,CrystalPlate,CrystalWell,EchoX,...,Deposition_PDB_ID,Deposition_PDB_file,Deposition_Date,Deposition_mmCIF_model_file,Deposition_mmCIF_SF_file,Label,table_one,AssayIC50,LastUpdated,LastUpdated_by
136,138,lb32627-66,DSiPoised_DMSO_Set2,M05,DSIPoised,CCOC(=O)c1ccoc1,Z293079056,9cq5_2023-10-09_RI1000-0276-3drop,B01d,-2046.0,...,None,None,None,None,None,None,None,None,30/11/2023 09:17,vfi61159
659,661,lb32627-66,DSiPoised_DMSO_Set1,I09,DSIPoised,CC(NC(=O)C)c1nnc2ccccn12,Z131516158,9cq0_2023-10-09_RI1000-0276-3drop,A12d,-1622.0,...,None,None,None,None,None,None,None,None,22/03/2024 11:39,vfi61159
2493,2495,lb32627-66,C22168492-P2-1873988-02,AE05,ASAPPTBOAM,COc1cncc(CCC(=O)N[C@H]2CCNC2=O)c1,ASAP-0029229-001,9epq_2024-07-17_RI1000-0276-3drop,C01a,8.0,...,None,None,None,None,None,None,None,None,20/09/2024 11:05,vfi61159
2945,2947,lb32627-66,1530852-Y4-339,R08,ASAPPTBOAM,Cc1noc(C)c1C[C@H](C)C(=O)N(C)C,ASAP-0030774-001,9euo_2024-08-21_RI1000-0276-3drop,A09a,-373.0,...,None,None,None,None,None,None,None,None,14/10/2024 13:59,vfi61159
2949,2951,lb32627-66,1530852-Y4-339,C09,ASAPPTBOAM,NS(=O)(=O)CCc1cccnc1F,ASAP-0030734-001,9euo_2024-08-21_RI1000-0276-3drop,C09c,252.0,...,None,None,None,None,None,None,None,None,20/09/2024 11:24,vfi61159
3012,3014,lb32627-66,1530852-Y4-339,D06,ASAPPTBOAM,COC(=O)c1ccncc1N,ASAP-0018631-002,9euo_2024-08-21_RI1000-0276-3drop,G12a,136.0,...,None,None,None,None,None,None,None,None,08/10/2024 09:47,vfi61159
3025,3027,lb32627-66,1530852-Y4-339,AB07,ASAPPTBOAM,CC(=O)Nc1cccc(NC(N)=O)c1,ASAP-0019110-002,9evz_2024-08-29_RI1000-0276-3drop,C01d,-852.0,...,7I8U,None,None,None,None,None,None,None,21/03/2025 08:43,vfi61159
3355,3357,lb32627-66,1530852-Y4-339,B09,ASAPPTBOAM,CN1CCCc2ccc(S(N)(=O)=O)cc21,ASAP-0018553-002,9exy_2024-09-11_RI1000-0080-3drop,E08d,-645.4,...,None,None,None,None,None,None,None,None,14/10/2024 14:29,vfi61159
3551,3553,lb32627-66,1530852-Y4-365,Y05,ASAPPTBOAM,Cn1nc(C(=O)NC[C@H](O)CC2CC2)c2ccccc21,ASAP-0031542-001,9ezt_2024-09-24_RI1000-0276-3drop,A09c,856.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:48,vfi61159


- Main Table

In [28]:
main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]

,ID,LabVisit,LibraryPlate,SourceWell,LibraryName,CompoundSMILES,CompoundCode,CrystalPlate,CrystalWell,EchoX,...,Deposition_PDB_ID,Deposition_PDB_file,Deposition_Date,Deposition_mmCIF_model_file,Deposition_mmCIF_SF_file,Label,table_one,AssayIC50,LastUpdated,LastUpdated_by
1908,1910,lb32627-66,EVA712A-P2-W1-1784480-02,AD06,ASAPPTBOAM,CCn1nc(-c2ccccc2)cc1NC(=O)C(C)(C)C(F)F,ASAP-0016710-001,9dy5_2024-03-07_RI1000-0276-3drop,A02d,-907.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:19,vfi61159
1909,1911,lb32627-66,EVA712A-P2-W1-1784480-02,C06,ASAPPTBOAM,O=C(Nc1cncc(F)n1)C1(c2cccc(Br)c2)CCOCC1,ASAP-0016714-001,9dy5_2024-03-07_RI1000-0276-3drop,B02a,-232.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:19,vfi61159
1918,1920,lb32627-66,EVA712A-P2-W1-1784480-02,A07,ASAPPTBOAM,CC(C)(C(=O)Nc1cccc2nc(Cl)ccc12)C1CCS(=O)(=O)CC1,ASAP-0016738-001,9dy5_2024-03-07_RI1000-0276-3drop,A05a,-148.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:20,vfi61159
1932,1934,lb32627-66,EVA712A-P2-W1-1784480-02,Z05,ASAPPTBOAM,Cc1cnccc1NC(=O)C(C)C,ASAP-0016705-001,9dy5_2024-03-07_RI1000-0276-3drop,A09a,-399.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:20,vfi61159
1933,1935,lb32627-66,EVA712A-P2-W1-1784480-02,B06,ASAPPTBOAM,CCCC(=O)Nc1c(-c2ccccc2)c(C)nn1C,ASAP-0016713-001,9dy5_2024-03-07_RI1000-0276-3drop,A09c,-438.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:20,vfi61159
1934,1936,lb32627-66,EVA712A-P2-W1-1784480-02,J06,ASAPPTBOAM,COCCc1cc(NC(=O)C(C)C)n(C)n1,ASAP-0016721-001,9dy5_2024-03-07_RI1000-0276-3drop,B09a,-372.0,...,None,None,None,None,None,None,None,None,13/09/2024 13:57,vfi61159
1935,1937,lb32627-66,EVA712A-P2-W1-1784480-02,Y06,ASAPPTBOAM,Cn1cc(-c2cc(N)cnn2)c2ccccc21,ASAP-0016736-001,9dy5_2024-03-07_RI1000-0276-3drop,B09c,-644.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:20,vfi61159
1999,2001,lb32627-66,XC-EVA712A-wasgo-iter-1.0-3,M3,ASAPTBOAM,Cc1nn(C)c(C)c1-c1ccc(N)cn1,ASAP-0022678,9e6q_2024-05-01_RI1000-0276-3drop,F01c,1400.0,...,None,None,None,None,None,None,None,None,13/09/2024 13:57,vfi61159
2002,2004,lb32627-66,XC-EVA712A-wasgo-iter-1.0-3,P3,ASAPTBOAM,Nc1cccnc1-c1cccc2cccnc12,ASAP-0022917,9e6q_2024-05-01_RI1000-0276-3drop,G01c,-148.0,...,None,None,None,None,None,None,None,None,13/09/2024 13:57,vfi61159
2018,2020,lb32627-66,XC-EVA712A-wasgo-iter-1.0-2,A1,ASAPTBOAM,Cc1ccnc(-c2c[nH]c3ccccc23)c1,ASAP-0023087,9e6q_2024-05-01_RI1000-0276-3drop,E02c,-319.0,...,None,None,None,None,None,None,None,None,15/12/2024 22:21,vfi61159


In [ ]:
# None found in the pandda_df
main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]["CrystalName"].isin(pandda_df["CrystalName"])

1908    False
1909    False
1918    False
1932    False
1933    False
1934    False
1935    False
1999    False
2002    False
2018    False
2154    False
2205    False
2220    False
Name: CrystalName, dtype: bool

In [41]:
# Only one repeating 
print(main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ].shape)
print(main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]["CompoundCode"].value_counts())

(13, 195)
CompoundCode
ASAP-0027905        2
ASAP-0016710-001    1
ASAP-0016738-001    1
ASAP-0016714-001    1
ASAP-0016705-001    1
ASAP-0016713-001    1
ASAP-0016736-001    1
ASAP-0016721-001    1
ASAP-0022678        1
ASAP-0022917        1
ASAP-0023087        1
ASAP-0027917        1
Name: count, dtype: int64


In [43]:
# Get ligands of apo structures -> ligands that did not bind -> negative control
print(main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]["CompoundSMILES"].unique())

['CCn1nc(-c2ccccc2)cc1NC(=O)C(C)(C)C(F)F'
 'O=C(Nc1cncc(F)n1)C1(c2cccc(Br)c2)CCOCC1'
 'CC(C)(C(=O)Nc1cccc2nc(Cl)ccc12)C1CCS(=O)(=O)CC1' 'Cc1cnccc1NC(=O)C(C)C'
 'CCCC(=O)Nc1c(-c2ccccc2)c(C)nn1C' 'COCCc1cc(NC(=O)C(C)C)n(C)n1'
 'Cn1cc(-c2cc(N)cnn2)c2ccccc21' 'Cc1nn(C)c(C)c1-c1ccc(N)cn1'
 'Nc1cccnc1-c1cccc2cccnc12' 'Cc1ccnc(-c2c[nH]c3ccccc23)c1'
 'CCCC(C)C(=O)Nc1cccc(-c2ccccc2)n1' 'c1ccc(OCCOc2ccnc(-c3ccccc3)c2)cc1']


In [64]:
# get pdb paths 
apo_dimple_paths = main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]["DimplePathToPDB"].values.tolist()
apo_cryst_name =  main_df[ ( (main_df["DimplePANDDAwasRun"] == "TRUE") & (main_df["DimplePANDDAhit"] == "FALSE") )  ]["CrystalName"].values.tolist()
# print(apo_cryst_namme.__len__())

In [66]:
# copy files to apo directory
apo_dir = rootdir.joinpath("data/ev2a/dls/apoprots")
for path_str, cryst_name in zip( apo_dimple_paths, apo_cryst_name):
    path = Path(path_str)
    save_file = apo_dir.joinpath( "{}_{}".format(cryst_name, path.name) )
    copy2(path, save_file)